# Formula 1 Race Winner Prediction

This notebook implements a Machine Learning pipeline to predict the winner of Formula 1 races.

**Steps covered:**
1. Define the Problem and Load the Data
2. Data Preprocessing
3. Exploratory Data Analysis (EDA)
4. Feature Engineering
5. Split the Data into Training and Testing Sets
6. **NEW**: Choose and Train Multiple Models
7. Hyperparameter Tuning (Optimization) of Best Model
8. Make Predictions
9. Evaluate the Model
10. Save the Best Model

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, precision_score, recall_score, f1_score

# Models
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier

# Ensure output directories exist
os.makedirs('out/images', exist_ok=True)
os.makedirs('models', exist_ok=True)

## 1. Load the Data
We load all the CSV files from the `data` directory.

In [ ]:
circuits = pd.read_csv('data/circuits.csv')
constructor_results = pd.read_csv('data/constructor_results.csv')
constructor_standings = pd.read_csv('data/constructor_standings.csv')
constructors = pd.read_csv('data/constructors.csv')
driver_standings = pd.read_csv('data/driver_standings.csv')
drivers = pd.read_csv('data/drivers.csv')
lap_times = pd.read_csv('data/lap_times.csv')
pit_stops = pd.read_csv('data/pit_stops.csv')
qualifying = pd.read_csv('data/qualifying.csv')
races = pd.read_csv('data/races.csv')
results = pd.read_csv('data/results.csv')
seasons = pd.read_csv('data/seasons.csv')
sprint_results = pd.read_csv('data/sprint_results.csv')
status = pd.read_csv('data/status.csv')

# Merge into a single dataframe
df = results.merge(races[['raceId', 'year', 'round', 'circuitId', 'date']], on='raceId')
df = df.merge(drivers[['driverId', 'driverRef', 'nationality', 'dob']], on='driverId')
df = df.merge(constructors[['constructorId', 'constructorRef', 'nationality']], on='constructorId', suffixes=('_driver', '_constructor'))
df = df.merge(circuits[['circuitId', 'circuitRef', 'country', 'lat', 'lng']], on='circuitId')
df = df.merge(status[['statusId', 'status']], on='statusId', how='left')

print(f"Main Dataframe Shape: {df.shape}")

## 2. Data Preprocessing
- Handle missing values (replace mapped `\N` with NaN)
- Convert data types (dates, numerics)

In [ ]:
# Replace '\N' with NaN
df.replace('\\N', np.nan, inplace=True)

# Convert date columns
df['date'] = pd.to_datetime(df['date'])
df['dob'] = pd.to_datetime(df['dob'])

# Convert numeric columns that might be strings
numeric_cols = ['grid', 'positionOrder', 'points', 'laps']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Fill NaNs if appropriate (e.g. 0 points)
df['points'].fillna(0, inplace=True)

print(df.info())

## 3. Exploratory Data Analysis (EDA)
Visualizing the data and saving plots to `out/images`.

In [ ]:
sns.set_palette("coolwarm")

# 1. Top 10 Winners
plt.figure(figsize=(12, 6))
top_winners = df[df['positionOrder'] == 1]['driverRef'].value_counts().head(10)
sns.barplot(x=top_winners.index, y=top_winners.values)
plt.title('Top 10 F1 Winners (All Time)')
plt.ylabel('Wins')
plt.xlabel('Driver')
plt.savefig('out/images/top_winners.png')
plt.show()

# 2. Grid Position vs Final Position Correlation
plt.figure(figsize=(10, 6))
correlation_data = df[['grid', 'positionOrder']].dropna()
sns.regplot(x='grid', y='positionOrder', data=correlation_data, scatter_kws={'alpha':0.1}, line_kws={'color':'red'})
plt.title('Grid Position vs Final Position')
plt.savefig('out/images/grid_pos_correlation.png')
plt.show()

# 3. Constructor Wins
plt.figure(figsize=(12, 6))
top_constructors = df[df['positionOrder'] == 1]['constructorRef'].value_counts().head(10)
sns.barplot(x=top_constructors.values, y=top_constructors.index, orient='h')
plt.title('Top 10 Constructors by Wins')
plt.savefig('out/images/constructor_wins.png')
plt.show()

## 4. Feature Engineering
Creating features for the model.
- `Driver Age`: Age of driver at race date
- `Target`: 1 if winner (`positionOrder` == 1), else 0

In [ ]:
# Calculate Driver Age
df['driver_age'] = (df['date'] - df['dob']).dt.days / 365.25

# Create Target Variable
df['is_winner'] = df['positionOrder'].apply(lambda x: 1 if x == 1 else 0)

# Encode Categorical Variables
le_driver = LabelEncoder()
le_constructor = LabelEncoder()
le_circuit = LabelEncoder()

df['driverId_enc'] = le_driver.fit_transform(df['driverRef'])
df['constructorId_enc'] = le_constructor.fit_transform(df['constructorRef'])
df['circuitId_enc'] = le_circuit.fit_transform(df['circuitRef'])

# Select Features
features = ['grid', 'points', 'laps', 'driverId_enc', 'constructorId_enc', 'circuitId_enc', 'driver_age']
target = 'is_winner'

df_model = df[features + [target, 'year']].dropna()

# Standardize features (important for SVM, KNN, Logistic Regression)
scaler = StandardScaler()
df_model_scaled = df_model.copy()
df_model_scaled[['grid', 'points', 'laps', 'driver_age']] = scaler.fit_transform(df_model[['grid', 'points', 'laps', 'driver_age']])

print(df_model_scaled.head())

## 5. Split Data
We use a temporal split to avoid data leakage (training on future races to predict past ones).
- **Train**: Races before 2023
- **Test**: Races in 2023 and after

In [ ]:
train = df_model_scaled[df_model_scaled['year'] < 2023]
test = df_model_scaled[df_model_scaled['year'] >= 2023]

X_train = train[features]
y_train = train[target]
X_test = test[features]
y_test = test[target]

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")

## 6. Choose and Train Models
We will test multiple algorithms:
1. Random Forest
2. XGBoost
3. Logistic Regression
4. Decision Tree
5. Naive Bayes

In [ ]:
models = {
    'Random Forest': RandomForestClassifier(random_state=42, class_weight='balanced'),
    'XGBoost': XGBClassifier(random_state=42, scale_pos_weight=10, eval_metric='logloss'), # scale_pos_weight for imbalance
    'Logistic Regression': LogisticRegression(random_state=42, class_weight='balanced', max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(random_state=42, class_weight='balanced'),
    'Naive Bayes': GaussianNB()
}

results_metrics = []

print("Training models...")
best_model_name = ""
best_score = 0

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else model.decision_function(X_test)
    
    roc_auc = roc_auc_score(y_test, y_prob)
    f1 = f1_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    
    results_metrics.append({
        'Model': name,
        'ROC AUC': roc_auc,
        'F1 Score': f1,
        'Precision': precision,
        'Recall': recall
    })
    
    print(f"{name}: ROC AUC = {roc_auc:.4f}, F1 = {f1:.4f}")
    
    if roc_auc > best_score:
        best_score = roc_auc
        best_model_name = name

results_df = pd.DataFrame(results_metrics)
print("\nModel Comparison:")
print(results_df.sort_values(by='ROC AUC', ascending=False))

## 7. Hyperparameter Tuning (Optimization)
We will tune the hyperparameters of the best performing model.

In [ ]:
print(f"\nOptimizing Best Model: {best_model_name}")
final_model = None

if best_model_name == 'Random Forest':
    param_grid = {
        'n_estimators': [100, 200, 300],
        'max_depth': [10, 20, None],
        'min_samples_split': [2, 5, 10]
    }
    grid_search = GridSearchCV(RandomForestClassifier(random_state=42, class_weight='balanced'), param_grid, cv=3, scoring='roc_auc', n_jobs=-1)

elif best_model_name == 'XGBoost':
    param_grid = {
        'n_estimators': [100, 200],
        'max_depth': [3, 6, 10],
        'learning_rate': [0.01, 0.1, 0.3]
    }
    grid_search = GridSearchCV(XGBClassifier(random_state=42, scale_pos_weight=10, eval_metric='logloss'), param_grid, cv=3, scoring='roc_auc', n_jobs=-1)

else: # Fallback for others or default to RF if simplified
    param_grid = {'C': [0.1, 1, 10]} if best_model_name == 'Logistic Regression' else {}
    # Assuming Logistic Regression or others for simplicity in this conditional block
    model_instance = models[best_model_name]
    grid_search = GridSearchCV(model_instance, param_grid, cv=3, scoring='roc_auc', n_jobs=-1)

grid_search.fit(X_train, y_train)
final_model = grid_search.best_estimator_
print(f"Best Parameters: {grid_search.best_params_}")

## 8 & 9. Final Evaluation
Evaluating the optimized model.

In [ ]:
y_pred_final = final_model.predict(X_test)
y_prob_final = final_model.predict_proba(X_test)[:, 1]

print(f"Final Model: {best_model_name}")
print("Classification Report:")
print(classification_report(y_test, y_pred_final))
print(f"Final ROC AUC: {roc_auc_score(y_test, y_prob_final):.4f}")

# Confusion Matrix
plt.figure(figsize=(6, 4))
sns.heatmap(confusion_matrix(y_test, y_pred_final), annot=True, fmt='d', cmap='Greens')
plt.title(f'Confusion Matrix ({best_model_name})')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.savefig('out/images/final_confusion_matrix.png')
plt.show()

## 10. Save Best Model
Saving the best model to the `models/` directory.

In [ ]:
model_data = {
    'model': final_model,
    'model_name': best_model_name,
    'le_driver': le_driver,
    'le_constructor': le_constructor,
    'le_circuit': le_circuit,
    'scaler': scaler,  # Don't forget the scaler!
    'feature_names': features
}

joblib.dump(model_data, 'models/best_f1_model.pkl')
print("Best model saved to 'models/best_f1_model.pkl'")